# 03 — Baseline Models

Implements and evaluates three baselines on walk-forward fold 4.
These are the benchmarks the LightGBM model must beat to justify deployment.

In [ ]:
import sys
sys.path.insert(0, '../..')

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ml.athena_client import get_s3_client, BUCKET
from ml.train import FOLDS, wape, rmse, bias, fva, evaluate_baselines

# load features
s3  = get_s3_client()
obj = s3.get_object(Bucket=BUCKET, Key='ml/features/features.parquet')
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))
df['date'] = pd.to_datetime(df['date'])
day0 = df['date'].min()
print(f'Features loaded: {len(df):,} rows')

## Evaluate all baselines on all folds

In [ ]:
results = []
for fold_idx, fold in enumerate(FOLDS, 1):
    metrics = evaluate_baselines(
        df, fold['train_end'], fold['val_start'], fold['val_end'], day0
    )
    for model_name, m in metrics.items():
        results.append({'fold': fold_idx, 'model': model_name, **m})

results_df = pd.DataFrame(results)
print('\nBaseline results by fold:')
print(results_df.pivot_table(index='model', columns='fold', values='wape').round(4))

In [ ]:
summary = results_df.groupby('model')[['wape', 'rmse', 'bias']].mean().round(4)
print('\nMean across all folds:')
print(summary)

# FVA reference: seasonal naive is the benchmark
sn_wape = summary.loc['seasonal_naive', 'wape']
summary['fva'] = summary['wape'] / sn_wape
print('\nFVA (relative to seasonal naive — lower is better):')
print(summary[['wape', 'fva']].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
models = ['naive', 'seasonal_naive', 'rolling_mean']
colors = ['steelblue', 'coral', 'seagreen']

for ax, metric in zip(axes, ['wape', 'rmse', 'bias']):
    vals = [results_df[results_df['model'] == m][metric].mean() for m in models]
    ax.bar(models, vals, color=colors, edgecolor='white')
    ax.set_title(metric.upper())
    ax.set_xticklabels(models, rotation=20)

plt.suptitle('Baseline model comparison (mean across folds)', y=1.02)
plt.tight_layout()
plt.show()

print('\nConclusion: seasonal_naive is the strongest baseline.')
print('The LightGBM model must achieve FVA < 1.0 to justify deployment.')